# 10 - Estatisticas Descritivas

## Objetivo
Explorar dados com funcoes estatisticas do Pandas.

## Conceitos

### describe
`df.describe()` retorna contagem, media, desvio padrao, min, quartis e max.
Para colunas categoricas, mostra contagem, unicos e moda.

### Medidas comuns
- `mean`, `median`, `mode`: tendencia central.
- `std`, `var`: dispersao.
- `min`, `max`, `quantile`: posicao.
- `sum`, `cumsum`: acumulados.
- `count`, `nunique`: contagens.

### Por eixo
Todas aceitam `axis` (0 = colunas, 1 = linhas).

### Correlacao e covariancia
- `df.corr()`: matriz de correlacao (Pearson por padrao).
- `df.corr(method="spearman")`: correlacao de postos.
- `df.cov()`: covariancia.
- `df.corrwith(series)`: correlacao com uma Series.

### Agrupamento estatistico
Combine `groupby` com `describe` para estatisticas por grupo.

### Janelas moveis
- `rolling(window)`: media/desvio moveis.
- `expanding()`: acumulado crescente.
- `ewm(span)`: media exponencial.

### Valores ausentes
Funcoes ignoram NaN por padrao. Use `skipna=False` para nao ignorar.

## DataFrame de exemplo
Geramos 100 registros aleatorios com `numpy.random` (semente fixa para
reprodutibilidade). Temos duas colunas numericas (`vendas`, `clientes`)
e uma categorica (`regiao`).

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    "vendas": np.random.normal(1000, 200, 100).round(2),
    "clientes": np.random.randint(10, 100, 100),
    "regiao": np.random.choice(["Norte", "Sul", "Leste"], 100),
})
print("Head:\n", df.head())

## describe
`describe()` resume cada coluna com contagem, media, desvio, min, quartis
e max. Podemos restringir a colunas numericas (`np.number`) ou categoricas
(`"object"`).

In [ ]:
# Estatisticas gerais
print("\ndescribe:\n", df.describe())

# Por tipo
print("\nSomente numericas:\n",
      df.describe(include=[np.number]))
print("\nSomente categoricas:\n",
      df.describe(include=["object"]))

## Medidas de tendencia central e dispersao
- `mean`, `median`: centro da distribuicao.
- `std`, `var`: dispersao (desvio padrao e variancia).
- `min`, `max`: extremos.

In [ ]:
# Medidas especificas
print("\nMedia:\n", df[["vendas", "clientes"]].mean())
print("Mediana:\n", df[["vendas", "clientes"]].median())
print("Desvio:\n", df[["vendas", "clientes"]].std())
print("Variancia:\n", df[["vendas", "clientes"]].var())
print("Minimo:\n", df[["vendas", "clientes"]].min())
print("Maximo:\n", df[["vendas", "clientes"]].max())

## Quantis, moda e contagens
- `quantile([...])`: quartis (ou outros percentis).
- IQR: intervalo interquartil (`Q3 - Q1`), mede a dispersao central.
- `mode()`: valor mais frequente.
- `value_counts()`: frequencia de cada valor categorico.
- `nunique()`: numero de valores distintos.

In [ ]:
# Quantis
print("\nQuartis de vendas:\n",
      df["vendas"].quantile([0.25, 0.5, 0.75]))
print("IQR:",
      df["vendas"].quantile(0.75) - df["vendas"].quantile(0.25))

# Moda
print("\nModa da regiao:\n", df["regiao"].mode())

# Contagens
print("\nContagem por regiao:\n", df["regiao"].value_counts())
print("Unicos:", df["regiao"].nunique())

## Acumulados
`cumsum()` retorna a soma acumulada ao longo do eixo. Util para
acompanhar totais progressivos.

In [ ]:
# Acumulados
print("\nCumsum de clientes (5 primeiros):\n",
      df["clientes"].cumsum().head())

## Correlacao e covariancia
- `corr()`: matriz de correlacao de Pearson (linear) por padrao.
- `corr(method="spearman")`: correlacao de postos (monotonica).
- `cov()`: matriz de covariancia.
- `corrwith(series)`: correlacao de cada coluna com uma Series externa.

In [ ]:
# Correlacao
print("\nCorrelacao:\n", df[["vendas", "clientes"]].corr())
print("Spearman:\n",
      df[["vendas", "clientes"]].corr(method="spearman"))
print("Covariancia:\n", df[["vendas", "clientes"]].cov())

# Correlacao com uma Series
print("\ncorrwith clientes:\n",
      df[["vendas", "clientes"]].corrwith(df["clientes"]))

## Estatisticas por grupo
Combinando `groupby` com `mean` ou `describe`, obtemos estatisticas
por categoria.

In [ ]:
# Estatisticas por grupo
print("\nPor regiao:\n",
      df.groupby("regiao")[["vendas", "clientes"]].mean())
print("\ndescribe por regiao:\n",
      df.groupby("regiao")["vendas"].describe())

## Janelas moveis
- `rolling(window).mean()`: media movel de tamanho fixo.
- `ewm(span).mean()`: media exponencial, dando mais peso aos pontos recentes.

Ordenamos o DataFrame por `vendas` antes, para ilustrar a suavizacao.

In [ ]:
# Janela movel
df_sorted = df.sort_values("vendas").reset_index(drop=True)
df_sorted["media_movel_5"] = df_sorted["vendas"].rolling(5).mean()
print("\nMedia movel:\n",
      df_sorted[["vendas", "media_movel_5"]].head(10))

# Media exponencial
df_sorted["ewm_10"] = df_sorted["vendas"].ewm(span=10).mean()
print("\nEWM:\n", df_sorted[["vendas", "ewm_10"]].head())

## skipna
Por padrao, as funcoes **ignoram** NaN. Com `skipna=False`, o resultado
e NaN se houver qualquer NaN na serie.

In [ ]:
# skipna
s = pd.Series([1.0, np.nan, 3.0])
print("\nskipna=True:", s.mean())
print("skipna=False:", s.mean(skipna=False))

## Tabela resumo
Consolidamos as principais estatisticas em um unico DataFrame,
facilitando a visualizacao lado a lado.

In [ ]:
# Tabela resumo
resumo = pd.DataFrame({
    "media": df[["vendas", "clientes"]].mean(),
    "mediana": df[["vendas", "clientes"]].median(),
    "desvio": df[["vendas", "clientes"]].std(),
    "min": df[["vendas", "clientes"]].min(),
    "max": df[["vendas", "clientes"]].max(),
})
print("\nResumo:\n", resumo)